# Pipeline de ML passo a passo

Este notebook reorganiza sua pipeline original em células sequenciais, sem classe e sem funções customizadas para o fluxo principal.
A ideia é executar **de cima para baixo**, uma célula por vez.

In [7]:
import pandas as pd
import numpy as np
import warnings
import unicodedata

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 1) Definir o caminho do arquivo CSV

In [8]:
file_path = "../data/raw/Dados Históricos - Ibovespa.csv"
train_start_date = pd.to_datetime("2020-01-01")
target_threshold = 0.003
test_size = 30
cutoff_proba = 0.45

## 2) Ler os dados brutos

In [9]:
df_raw = pd.read_csv(file_path)
df_raw.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,10.03.2026,183.447,180.921,185.324,180.693,"10,08B","1,40%"
1,09.03.2026,180.915,179.367,181.952,177.637,"11,42B","0,86%"
2,06.03.2026,179.365,180.463,181.091,178.556,"9,92B","-0,61%"
3,05.03.2026,180.464,185.365,185.366,179.895,"10,60B","-2,64%"
4,04.03.2026,185.366,183.110,186.306,183.110,"9,11B","1,24%"


## 3) Limpar os nomes das colunas

In [10]:
def strip_accents(s):
    return ''.join(
        ch for ch in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(ch)
    )

def clean_col(c):
    return strip_accents(str(c)).replace('.', '').replace('%', '').lower().strip()

cols = {clean_col(c): c for c in df_raw.columns}
cols

{'data': 'Data',
 'ultimo': 'Último',
 'abertura': 'Abertura',
 'maxima': 'Máxima',
 'minima': 'Mínima',
 'vol': 'Vol.',
 'var': 'Var%'}

## 4) Renomear as colunas principais para um padrão único

In [11]:
rename = {}

for k, v in cols.items():
    if "data" in k:
        rename[v] = "Date"
    elif "ultimo" in k or "fechamento" in k:
        rename[v] = "Close"
    elif "abert" in k:
        rename[v] = "Open"
    elif "max" in k:
        rename[v] = "High"
    elif "min" in k:
        rename[v] = "Low"
    elif "vol" in k:
        rename[v] = "Volume"

rename

{'Data': 'Date',
 'Último': 'Close',
 'Abertura': 'Open',
 'Máxima': 'High',
 'Mínima': 'Low',
 'Vol.': 'Volume'}

## 5) Padronizar os formatos numéricos

In [12]:
df = df_raw.rename(columns=rename).copy()

def coerce_pt_number(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace('.', '').replace(',', '.')
    try:
        return float(s)
    except:
        return np.nan

def parse_pt_volume(v):
    if pd.isna(v):
        return np.nan

    s = str(v).strip()
    mult = 1

    if s.endswith(("M", "m")):
        mult, s = 1e6, s[:-1]
    elif s.endswith(("K", "k")):
        mult, s = 1e3, s[:-1]
    elif s.endswith(("B", "b")):
        mult, s = 1e9, s[:-1]

    try:
        return float(s.replace('.', '').replace(',', '.')) * mult
    except:
        return np.nan

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for col in ["Close", "Open", "High", "Low"]:
    if col in df.columns:
        df[col] = df[col].apply(coerce_pt_number)

if "Volume" in df.columns:
    df["Volume"] = df["Volume"].apply(parse_pt_volume)

df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")
df.head()

,Close,Open,High,Low,Volume,Var%
Date,,,,,,
2006-01-02,33507.0,33462.0,33519.0,3286.0,72320000.0,"0,15%"
2006-01-03,34541.0,33507.0,34563.0,33507.0,148380000.0,"3,09%"
2006-01-04,35002.0,3454.0,35223.0,3454.0,147130000.0,"1,33%"
2006-01-05,34936.0,35006.0,35088.0,34681.0,142070000.0,"-0,19%"
2006-01-06,35475.0,3517.0,35529.0,3494.0,115090000.0,"1,54%"


## 6) Conferir tipos e valores ausentes

In [13]:
print(df.dtypes)
print()
print(df.isna().sum())

Close     float64
Open      float64
High      float64
Low       float64
Volume    float64
Var%          str
dtype: object

Close     0
Open      0
High      0
Low       0
Volume    1
Var%      0
dtype: int64


## 7) Criar a variável alvo (target)

Aqui o alvo é:
- 1 = o retorno do próximo dia foi maior que o limiar
- 0 = caso contrário

In [14]:
df["Next_Close"] = df["Close"].shift(-1)
df["Retorno_Pct"] = df["Close"].pct_change().shift(-1)
df["Target"] = (df["Retorno_Pct"] > target_threshold).astype(int)

df[["Close", "Next_Close", "Retorno_Pct", "Target"]].head(10)

,Close,Next_Close,Retorno_Pct,Target
Date,,,,
2006-01-02,33507.0,34541.0,0.030859,1
2006-01-03,34541.0,35002.0,0.013346,1
2006-01-04,35002.0,34936.0,-0.001886,0
2006-01-05,34936.0,35475.0,0.015428,1
2006-01-06,35475.0,35337.0,-0.003890,0
2006-01-09,35337.0,35049.0,-0.008150,0
2006-01-10,35049.0,35952.0,0.025764,1
2006-01-11,35952.0,35779.0,-0.004812,0
2006-01-12,35779.0,35897.0,0.003298,1


## 8) Criar variáveis de retorno e lags

In [15]:
df["Ret"] = df["Close"].pct_change() * 100

for k in [1, 2, 3, 5, 10]:
    df[f"Ret_lag_{k}"] = df["Ret"].shift(k)

df.filter(like="Ret").head(10)

,Retorno_Pct,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10
Date,,,,,,,
2006-01-02,0.030859,NaN,NaN,NaN,NaN,NaN,NaN
2006-01-03,0.013346,3.085922,NaN,NaN,NaN,NaN,NaN
2006-01-04,-0.001886,1.334646,3.085922,NaN,NaN,NaN,NaN
2006-01-05,0.015428,-0.188561,1.334646,3.085922,NaN,NaN,NaN
2006-01-06,-0.003890,1.542821,-0.188561,1.334646,3.085922,NaN,NaN
2006-01-09,-0.008150,-0.389006,1.542821,-0.188561,1.334646,NaN,NaN
2006-01-10,0.025764,-0.815010,-0.389006,1.542821,-0.188561,3.085922,NaN
2006-01-11,-0.004812,2.576393,-0.815010,-0.389006,1.542821,1.334646,NaN
2006-01-12,0.003298,-0.481197,2.576393,-0.815010,-0.389006,-0.188561,NaN


## 9) Criar médias móveis e distâncias

In [16]:
df["SMA_5"] = df["Close"].rolling(5).mean()
df["SMA_10"] = df["Close"].rolling(10).mean()
df["SMA_20"] = df["Close"].rolling(20).mean()

df["EMA_10"] = df["Close"].ewm(span=10).mean()
df["EMA_20"] = df["Close"].ewm(span=20).mean()

df["Dist_SMA5"] = (df["Close"] - df["SMA_5"]) / df["SMA_5"]
df["Dist_SMA20"] = (df["Close"] - df["SMA_20"]) / df["SMA_20"]

df[["SMA_5", "SMA_10", "SMA_20", "EMA_10", "EMA_20", "Dist_SMA5", "Dist_SMA20"]].tail()

,SMA_5,SMA_10,SMA_20,EMA_10,EMA_20,Dist_SMA5,Dist_SMA20
Date,,,,,,,
2026-03-04,187514.0,171588.7,170290.30,176722.149061,171564.366994,-0.011455,0.088529
2026-03-05,185405.8,170781.7,170029.80,177402.485596,172411.951090,-0.026654,0.061367
2026-03-06,183521.4,169664.8,169912.65,177759.306397,173074.146224,-0.022648,0.055631
2026-03-09,181843.0,168871.0,169852.05,178333.068870,173820.894203,-0.005103,0.065133
2026-03-10,181911.4,185300.8,178109.65,179262.874530,174737.666183,0.008441,0.029967


## 10) Criar variáveis de momentum e volatilidade

In [17]:
df["Momentum_3"] = df["Close"] - df["Close"].shift(3)
df["Momentum_7"] = df["Close"] - df["Close"].shift(7)
df["Momentum_14"] = df["Close"] - df["Close"].shift(14)

df["Vol_5"] = df["Close"].pct_change().rolling(5).std()
df["Vol_10"] = df["Close"].pct_change().rolling(10).std()
df["Vol_20"] = df["Close"].pct_change().rolling(20).std()

df[["Momentum_3", "Momentum_7", "Momentum_14", "Vol_5", "Vol_10", "Vol_20"]].tail()

,Momentum_3,Momentum_7,Momentum_14,Vol_5,Vol_10,Vol_20
Date,,,,,,
2026-03-04,-3421.0,-3487.0,-563.0,0.017200,2.887961,2.840462
2026-03-05,-8843.0,161315.0,-9235.0,0.019018,2.889209,2.841104
2026-03-06,-3740.0,-11882.0,-8401.0,0.019144,2.889723,2.840870
2026-03-09,-4451.0,-10090.0,-5549.0,0.020275,2.889184,2.840775
2026-03-10,2983.0,-5340.0,-2569.0,0.017017,2.843501,2.818992


## 11) Criar RSI

In [18]:
delta = df["Close"].diff()
up = delta.clip(lower=0).rolling(14).mean()
down = (-delta.clip(upper=0)).rolling(14).mean()
rs = up / down
df["RSI_14"] = 100 - (100 / (1 + rs))

df[["Close", "RSI_14"]].tail()

,Close,RSI_14
Date,,
2026-03-04,185366.0,49.923275
2026-03-05,180464.0,48.745343
2026-03-06,179365.0,48.856057
2026-03-09,180915.0,49.244917
2026-03-10,183447.0,49.652393


## 12) Criar MACD

In [19]:
ema_fast = df["Close"].ewm(span=12).mean()
ema_slow = df["Close"].ewm(span=26).mean()
df["MACD"] = ema_fast - ema_slow

df[["Close", "MACD"]].tail()

,Close,MACD
Date,,
2026-03-04,185366.0,6728.733955
2026-03-05,180464.0,6624.247625
2026-03-06,179365.0,6379.225599
2026-03-09,180915.0,6238.205771
2026-03-10,183447.0,6258.612392


## 13) Criar Bollinger %B

In [20]:
bb_mid = df["Close"].rolling(20).mean()
bb_std = df["Close"].rolling(20).std()
bb_up = bb_mid + 2 * bb_std
bb_low = bb_mid - 2 * bb_std

df["BB_pctB"] = (df["Close"] - bb_low) / (bb_up - bb_low)

df[["Close", "BB_pctB"]].tail()

,Close,BB_pctB
Date,,
2026-03-04,185366.0,0.572608
2026-03-05,180464.0,0.550320
2026-03-06,179365.0,0.545607
2026-03-09,180915.0,0.553392
2026-03-10,183447.0,0.535506


## 14) Criar Stochastic %K

In [21]:
lowest_low = df["Low"].rolling(14).min()
highest_high = df["High"].rolling(14).max()

df["StochK"] = 100 * (df["Close"] - lowest_low) / (highest_high - lowest_low)

df[["Close", "StochK"]].tail()

,Close,StochK
Date,,
2026-03-04,185366.0,96.195159
2026-03-05,180464.0,93.625398
2026-03-06,179365.0,93.049272
2026-03-09,180915.0,93.861824
2026-03-10,183447.0,95.189167


## 15) Criar variáveis extras de tendência, candle e calendário

In [22]:
df["Trend"] = (df["SMA_5"] > df["SMA_20"]).astype(int)
df["Range"] = df["High"] - df["Low"]
df["Body"] = df["Close"] - df["Open"]
df["dow"] = df.index.dayofweek
df["month"] = df.index.month

df[["Trend", "Range", "Body", "dow", "month"]].tail()

,Trend,Range,Body,dow,month
Date,,,,,
2026-03-04,1,167995.0,167055.0,2,3
2026-03-05,1,5471.0,-4901.0,3,3
2026-03-06,1,2535.0,-1098.0,4,3
2026-03-09,1,4315.0,1548.0,0,3
2026-03-10,1,4631.0,2526.0,1,3


## 16) Remover linhas com valores ausentes gerados pelos cálculos

In [23]:
df_model = df.dropna().copy()
df_model.shape

(4978, 37)

## 17) Separar X e y

In [24]:
X = df_model.drop(
    columns=["Open", "High", "Low", "Close", "Next_Close", "Target", "Retorno_Pct"],
    errors="ignore"
)

X = X.select_dtypes(exclude="object")
y = df_model["Target"]

print(X.shape)
print(y.shape)
X.head()

(4978, 29)
(4978,)


,Volume,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10,SMA_5,SMA_10,SMA_20,...,Vol_20,RSI_14,MACD,BB_pctB,StochK,Trend,Range,Body,dow,month
Date,,,,,,,,,,,,,,,,,,,,,
2006-01-31,137330000.0,0.368705,1.110465,-0.505077,10064.171123,-0.171686,1.774522,30567.0,33494.4,34472.3,...,22.515736,52.123837,21.252944,0.620608,98.807332,0,34693.0,140.0,1,1
2006-02-01,263910000.0,0.265743,0.368705,1.110465,-0.505077,-98.979035,-1.135928,38189.2,33731.0,34669.5,...,22.516063,51.630218,303.889947,0.616956,99.095446,1,-37501.0,103.0,2,2
2006-02-02,166780000.0,-3.068728,0.265743,0.368705,1.110465,10064.171123,-0.869349,38047.2,33880.9,34784.6,...,22.516577,50.968906,439.250904,0.577027,95.746611,1,1271.0,-1178.0,3,2
2006-02-03,115440000.0,-0.112588,-3.068728,0.265743,0.368705,-0.505077,2.940930,37935.2,33921.3,34900.9,...,22.516568,50.868089,536.310821,0.572022,95.627517,1,1083.0,-42.0,4,2
2006-02-06,130710000.0,0.158338,-0.112588,-3.068728,0.265743,1.110465,-0.442238,37751.0,33983.9,34993.2,...,22.516729,50.504209,609.582855,0.570858,95.794817,1,547.0,59.0,0,2


## 18) Fazer o corte temporal entre treino e teste

In [25]:
X_train = X.iloc[:-test_size].copy()
y_train = y.iloc[:-test_size].copy()

X_test = X.iloc[-test_size:].copy()
y_test = y.iloc[-test_size:].copy()

X_train = X_train[X_train.index >= train_start_date]
y_train = y_train[y_train.index >= train_start_date]

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)

Treino: (1510, 29) (1510,)
Teste: (30, 29) (30,)


## 19) Padronizar os dados

In [26]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index,
    columns=X_test.columns
)

X_train_scaled.head()

,Volume,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10,SMA_5,SMA_10,SMA_20,...,Vol_20,RSI_14,MACD,BB_pctB,StochK,Trend,Range,Body,dow,month
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,-0.440841,-0.164234,-0.167496,-0.167303,-0.165592,-0.165496,9.671829,0.522566,0.535214,0.091210,...,2.025672,0.091175,1.176230,0.431565,0.223966,0.924682,0.036197,0.067816,0.710394,-1.581623
2020-01-03,-0.440379,-0.167465,-0.164232,-0.167493,-0.167303,-0.164828,-0.166885,0.540674,0.569060,0.119923,...,2.025693,0.081802,1.188599,0.379028,0.220563,0.924682,0.004977,-0.011204,1.418911,-1.581623
2020-01-06,-0.440451,-0.167440,-0.167463,-0.164230,-0.167493,-0.164304,-0.165659,0.537482,0.593885,0.145832,...,2.025704,0.063445,1.170423,0.330326,0.217963,0.924682,0.004701,-0.010618,-1.415157,-1.581623
2020-01-07,-0.440926,-0.166925,-0.167438,-0.167461,-0.164230,-0.166070,-0.164802,0.538739,0.607555,0.166636,...,2.025724,3.484073,1.138866,0.308802,0.217286,0.924682,-0.002229,0.002329,-0.706640,-1.581623
2020-01-08,-0.440633,-0.167094,-0.166923,-0.167435,-0.167461,-0.166267,-0.165589,0.544651,0.614055,0.185033,...,2.025734,1.051352,1.093826,0.281265,0.215984,0.924682,0.009025,-0.002064,0.001877,-1.581623


## 20) Treinar Random Forest

In [27]:
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=1,
    max_features=0.7,
    max_samples=0.8,
    bootstrap=True,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

proba_rf = rf.predict_proba(X_test_scaled)[:, 1]
pred_rf = (proba_rf >= cutoff_proba).astype(int)

acc_rf = accuracy_score(y_test, pred_rf)
hits_rf = (y_test == pred_rf).sum()

print(f"RandomForest: {acc_rf * 100:.2f}% | {hits_rf}/{len(y_test)}")

RandomForest: 66.67% | 20/30


## 21) Treinar XGBoost

In [28]:
xgb = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_lambda=1,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_scaled, y_train)

proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
pred_xgb = (proba_xgb >= cutoff_proba).astype(int)

acc_xgb = accuracy_score(y_test, pred_xgb)
hits_xgb = (y_test == pred_xgb).sum()

print(f"XGBoost: {acc_xgb * 100:.2f}% | {hits_xgb}/{len(y_test)}")

XGBoost: 76.67% | 23/30


## 22) Treinar SVM

In [29]:
svm = SVC(
    kernel="rbf",
    C=20,
    gamma=0.05,
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm.fit(X_train_scaled, y_train)

proba_svm = svm.predict_proba(X_test_scaled)[:, 1]
pred_svm = (proba_svm >= cutoff_proba).astype(int)

acc_svm = accuracy_score(y_test, pred_svm)
hits_svm = (y_test == pred_svm).sum()

print(f"SVM: {acc_svm * 100:.2f}% | {hits_svm}/{len(y_test)}")

SVM: 70.00% | 21/30


## 23) Comparar os modelos

In [30]:
results = {
    "RandomForest": (acc_rf * 100, hits_rf, len(y_test), rf),
    "XGBoost": (acc_xgb * 100, hits_xgb, len(y_test), xgb),
    "SVM": (acc_svm * 100, hits_svm, len(y_test), svm),
}

for name, (acc, hits, total, _) in results.items():
    print(f"{name}: {acc:.2f}% | {hits}/{total}")

RandomForest: 66.67% | 20/30
XGBoost: 76.67% | 23/30
SVM: 70.00% | 21/30


## 24) Selecionar o melhor modelo

In [31]:
best_name = max(results.items(), key=lambda x: x[1][0])[0]
best_model = results[best_name][3]

print("Melhor modelo:", best_name)

Melhor modelo: XGBoost


## 25) Gerar o classification report do melhor modelo

In [32]:
proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
pred_best = (proba_best >= cutoff_proba).astype(int)

print(classification_report(y_test, pred_best))

              precision    recall  f1-score   support

           0       0.79      0.83      0.81        18
           1       0.73      0.67      0.70        12

    accuracy                           0.77        30
   macro avg       0.76      0.75      0.75        30
weighted avg       0.76      0.77      0.76        30



## 26) Conferir as previsões finais

In [33]:
df_resultado = pd.DataFrame({
    "y_real": y_test,
    "proba": proba_best,
    "y_pred": pred_best
}, index=y_test.index)

df_resultado.tail(30)

,y_real,proba,y_pred
Date,,,
2026-01-23,0,0.707001,1
2026-01-26,1,0.894864,1
2026-01-27,1,0.621334,1
2026-01-28,0,0.575620,1
2026-01-29,0,0.297524,0
2026-01-30,1,0.671769,1
2026-02-02,1,0.133024,0
2026-02-03,0,0.065049,0
2026-02-04,0,0.135343,0
